# Run MeanFlow on Google Colab GPU

Before running the cells: in Colab choose **Runtime > Change runtime type > GPU**.

If this notebook is the only file in Colab, zip your whole `MeanFlow` folder locally and upload that zip in the setup cell below.

In [ ]:
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "train.py").exists():
    print("train.py was not found in the current Colab folder.")
    print("Upload a zip of your full MeanFlow folder now, then this cell will unzip it.")
    from google.colab import files
    import zipfile

    uploaded = files.upload()
    zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_files:
        raise RuntimeError("Please upload a .zip file containing train.py, meanflow.py, model.py, utils.py, and data/.")

    zip_path = Path(zip_files[0])
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall("/content")

    matches = list(Path("/content").rglob("train.py"))
    if not matches:
        raise RuntimeError("Could not find train.py after unzipping.")

    project_root = matches[0].parent
    %cd {project_root}
else:
    print(f"Using project folder: {project_root}")

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!pip install -r requirements.txt

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. In Colab, set Runtime > Change runtime type > GPU, then rerun.")

CUDA available: True
GPU: Tesla T4


## Short GPU smoke test

This uses the included `data/smoke_test.png` and writes outputs to `outputs/colab_smoke`.

In [ ]:
!python train.py \
  --image data/smoke_test.png \
  --steps 100 \
  --batch-size 8 \
  --sample-every 50 \
  --checkpoint-every 100 \
  --output-dir outputs/colab_smoke

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for path in sorted(Path("outputs/colab_smoke").glob("sample_step_*.png")):
    print(path)
    display(Image(filename=str(path)))

## Longer one-image run

Run this after the smoke test works. Increase `--steps` if you want a longer overfit experiment.

In [ ]:
!python train.py \
  --image data/overfit1/imagenette_one.png \
  --steps 5000 \
  --batch-size 8 \
  --sample-every 500 \
  --checkpoint-every 1000 \
  --output-dir outputs/imagenette_one_overfit

## Optional: save outputs to Google Drive

Run this if you want to keep checkpoints after the Colab session ends.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs /content/drive/MyDrive/MeanFlow_outputs